# Benchmark: Deep Learning Baselines

**Models** : Vanilla LSTM · CNN-LSTM  
**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

These baselines share the **same Bidirectional DRNN architecture** as the  
proposed NiOA-optimised model, but use **fixed, manually chosen hyperparameters**  
rather than NiOA-tuned ones.  This isolates the contribution of the  
optimisation algorithm from the architecture itself.

Splits are loaded from the canonical frozen arrays.

### v2 changes
- **FATAL bug fixed**: `_m["sMAPE"]` → `_m_res["sMAPE"]` in both print statements.
- **OOM bug fixed**: `.predict(X_test)` replaced with batched prediction via  
  `SequenceDataGenerator` — X_test is 3.28 GB and causes GPU OOM if passed whole.
- `log_transform=TARGET_LOG_TRANSFORM` added to both `save_benchmark_results` calls.
- Plots produced in original kWh space (after expm1 inversion).
- Models saved as `.h5` + training configs saved as JSON for full reproducibility.


In [ ]:
import os, sys, json, gc, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow.keras.backend as K
from datetime import datetime
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import (
    Input, Conv1D, MaxPooling1D,
    LSTM, Dropout, Dense
)
from tensorflow.keras.callbacks import EarlyStopping

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'core')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from core.config import *
from core.config  import (
    SPLITS_DIR, BENCHMARK_DIR, RESULTS_DIR,
    FORECAST_HORIZONS, RANDOM_SEED,
    TARGET_LOG_TRANSFORM, LOSS_FUNCTION, HUBER_DELTA,
)
from core.utils   import configure_gpu, set_seeds, SequenceDataGenerator
from benchmarking.utils.data_loader import load_splits
from benchmarking.utils.metrics     import compute_metrics, save_benchmark_results, build_comparison_table

configure_gpu()
set_seeds(RANDOM_SEED)
sns.set(style='whitegrid')

ANALYSIS_DIR = os.path.join(RESULTS_DIR, 'analysis')
os.makedirs(ANALYSIS_DIR, exist_ok=True)

print(f'TensorFlow       : {tf.__version__}')
print(f'Project root     : {PROJECT_ROOT}')
print(f'log_transform    : {TARGET_LOG_TRANSFORM}')
print(f'Loss function    : {LOSS_FUNCTION}  delta={HUBER_DELTA}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU(s) found     : {len(gpus)}  →  {[g.name for g in gpus]}')

In [ ]:
# ── Edit these lines only ───────────────────────────────────────────────────
HORIZONS_TO_RUN = [60, 300, 900, 1800]
SKIP_COMPLETED  = False
BATCH_SIZE      = 16   # reduce to 8 if GPU has < 4 GB VRAM
EPOCHS          = 40
PATIENCE        = 7
# ───────────────────────────────────────────────────────────────────────────

for _h in HORIZONS_TO_RUN:
    assert _h in FORECAST_HORIZONS

print(f'Horizons queued : {HORIZONS_TO_RUN}')
print(f'Batch size      : {BATCH_SIZE}')
print(f'Epochs          : {EPOCHS}  patience={PATIENCE}')

---
## Main Loop — Vanilla LSTM and CNN-LSTM per horizon

In [ ]:
all_results = {}
loop_start  = time.time()

for _loop_idx, HORIZON in enumerate(HORIZONS_TO_RUN):

    _elapsed = (time.time() - loop_start) / 3600
    print(f'\n' + '=' * 60)
    print(f'  Horizon {_loop_idx+1}/{len(HORIZONS_TO_RUN)} : '
          f'k = {HORIZON} s   {_elapsed:.2f} h elapsed')
    print('=' * 60)

    if SKIP_COMPLETED:
        _done = all(
            os.path.isfile(
                os.path.join(BENCHMARK_DIR, f'horizon_{HORIZON}', _mdl, 'metrics.json')
            )
            for _mdl in ['vanilla_lstm', 'cnn_lstm']
        )
        if _done:
            print(f'  [SKIP] Both DL models already evaluated.')
            continue

    # ── Output directories ───────────────────────────────────────────────────
    _horizon_dir = os.path.join(BENCHMARK_DIR, f'horizon_{HORIZON}')
    _plots_dir   = os.path.join(_horizon_dir, 'dl_plots')
    _models_dir  = os.path.join(_horizon_dir, 'dl_models')
    for _d in [_horizon_dir, _plots_dir, _models_dir]:
        os.makedirs(_d, exist_ok=True)

    # ── Load splits ─────────────────────────────────────────────────────────
    print('  Loading splits ...')
    X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
        load_splits(SPLITS_DIR, HORIZON)

    SEQ_LEN   = X_train.shape[1]
    NUM_FEATS = X_train.shape[2]

    print(f'  seq={SEQ_LEN}  feats={NUM_FEATS}  '
          f'train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}')
    print(f'  RAM (train seq) ≈ {X_train.nbytes/1e9:.2f} GB   '
          f'GPU per batch ≈ {BATCH_SIZE*SEQ_LEN*NUM_FEATS*4/1e6:.1f} MB')

    # SequenceDataGenerator — feeds one batch at a time to GPU
    train_gen = SequenceDataGenerator(
        X_train, y_train, batch_size=BATCH_SIZE, shuffle=True,  seed=RANDOM_SEED
    )
    val_gen   = SequenceDataGenerator(
        X_val,   y_val,   batch_size=BATCH_SIZE, shuffle=False
    )

    # ── Loss function (same as NiOA-DRNN for fair comparison) ───────────────
    _loss = (
        tf.keras.losses.Huber(delta=HUBER_DELTA)
        if LOSS_FUNCTION == 'huber' else LOSS_FUNCTION
    )

    # ── Helper: save plots in kWh space ─────────────────────────────────────
    def _save_dl_plots(history, y_true_log1p, y_pred_log1p, model_name):
        """
        Converts predictions from log1p to kWh before plotting so all
        benchmark plots are directly comparable with NiOA-DRNN plots.
        """
        if TARGET_LOG_TRANSFORM:
            _yt = np.expm1(np.clip(y_true_log1p.astype(np.float64), -10., 30.))
            _yp = np.maximum(
                np.expm1(np.clip(y_pred_log1p.astype(np.float64), -10., 30.)), 0.
            )
        else:
            _yt = y_true_log1p
            _yp = np.maximum(y_pred_log1p, 0.)

        # Loss curve
        _fig, _ax = plt.subplots(figsize=(8, 4))
        _ax.plot(history.history['loss'],     lw=1.5, label='Train loss')
        _ax.plot(history.history['val_loss'], lw=1.5, label='Val loss')
        _ax.set_xlabel('Epoch')
        _ax.set_ylabel(f'{LOSS_FUNCTION.upper()} loss')
        _ax.set_title(f'{model_name} Training/Validation Loss  k={HORIZON}s')
        _ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(_plots_dir, f'{model_name}_loss.png'), dpi=180)
        plt.close('all')

        # Scatter (kWh)
        _fig, _ax = plt.subplots(figsize=(5, 5))
        _ax.scatter(_yt, _yp, alpha=0.3, s=4, color='steelblue')
        _lim = [min(_yt.min(), _yp.min()), max(_yt.max(), _yp.max())]
        _ax.plot(_lim, _lim, 'r--', lw=1.5)
        _ax.set_xlabel(f'Actual ΔE  k={HORIZON}s (kWh)')
        _ax.set_ylabel(f'Predicted ΔE  k={HORIZON}s (kWh)')
        _ax.set_title(f'{model_name} Predicted vs Actual  k={HORIZON}s')
        plt.tight_layout()
        plt.savefig(os.path.join(_plots_dir, f'{model_name}_scatter.png'), dpi=180)
        plt.close('all')

        # Residuals (kWh)
        _fig, _ax = plt.subplots(figsize=(7, 3))
        sns.histplot(_yt - _yp, bins=50, kde=True, ax=_ax, color='steelblue')
        _ax.axvline(0, color='red', lw=1.2, linestyle='--')
        _ax.set_xlabel('Residual (kWh)')
        _ax.set_title(f'{model_name} Residuals  k={HORIZON}s')
        plt.tight_layout()
        plt.savefig(os.path.join(_plots_dir, f'{model_name}_residuals.png'), dpi=180)
        plt.close('all')

    _horizon_results = {}

    # ────────────────────────────────────────────────────────────────────────
    # 1. Vanilla LSTM
    # ────────────────────────────────────────────────────────────────────────
    print('\n  [1/2] Vanilla LSTM ...')
    K.clear_session();  gc.collect()

    _m_lstm = Sequential(name='Vanilla_LSTM')
    _m_lstm.add(Input(shape=(SEQ_LEN, NUM_FEATS)))
    _m_lstm.add(LSTM(64, return_sequences=True))
    _m_lstm.add(Dropout(0.3))
    _m_lstm.add(LSTM(64, return_sequences=False))
    _m_lstm.add(Dropout(0.3))
    _m_lstm.add(Dense(32, activation='relu'))
    _m_lstm.add(Dense(1))
    _m_lstm.compile(loss=_loss, optimizer='adam', metrics=['mae'])

    _hist_lstm = _m_lstm.fit(
        train_gen,
        validation_data = val_gen,
        epochs          = EPOCHS,
        callbacks       = [EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1,
        )],
        verbose=1,
    )

    # Batched prediction — avoids GPU OOM on 3.28 GB X_test
    _test_gen_lstm = SequenceDataGenerator(
        X_test, X_test[:, 0, 0], batch_size=BATCH_SIZE, shuffle=False
    )
    _pred_lstm = _m_lstm.predict(_test_gen_lstm, verbose=0).flatten()[:len(X_test)]

    _m_res = save_benchmark_results(
        'vanilla_lstm', HORIZON, y_test, _pred_lstm,
        BENCHMARK_DIR, log_transform=TARGET_LOG_TRANSFORM,
        extra_meta={'epochs_trained': len(_hist_lstm.history['loss'])}
    )
    _save_dl_plots(_hist_lstm, y_test, _pred_lstm, 'vanilla_lstm')

    # Save model and training history
    _m_lstm.save(os.path.join(_models_dir, f'vanilla_lstm_k{HORIZON}.h5'))
    with open(os.path.join(_models_dir, f'vanilla_lstm_k{HORIZON}_history.json'), 'w') as _f:
        json.dump(
            {k: [float(v) for v in vals]
             for k, vals in _hist_lstm.history.items()}, _f, indent=2
        )

    _horizon_results['vanilla_lstm'] = _m_res
    # FIXED: was _m["sMAPE"] — _m is undefined here, correct variable is _m_res
    print(f'       MAE={_m_res["MAE"]:.6f}  RMSE={_m_res["RMSE"]:.6f}  '
          f'R2={_m_res["R2"]:.4f}  sMAPE={_m_res["sMAPE"]:.4f}')

    del _m_lstm, _hist_lstm, _pred_lstm, _test_gen_lstm
    K.clear_session();  gc.collect()

    # ────────────────────────────────────────────────────────────────────────
    # 2. CNN-LSTM
    # ────────────────────────────────────────────────────────────────────────
    print('\n  [2/2] CNN-LSTM ...')
    K.clear_session();  gc.collect()

    # Re-create generators — on_epoch_end may have shuffled the LSTM ones
    train_gen2 = SequenceDataGenerator(
        X_train, y_train, batch_size=BATCH_SIZE, shuffle=True, seed=RANDOM_SEED
    )
    val_gen2 = SequenceDataGenerator(
        X_val, y_val, batch_size=BATCH_SIZE, shuffle=False
    )

    # Rebuild loss object (Keras may have cleared it with the session)
    _loss2 = (
        tf.keras.losses.Huber(delta=HUBER_DELTA)
        if LOSS_FUNCTION == 'huber' else LOSS_FUNCTION
    )

    _m_cnn = Sequential(name='CNN_LSTM')
    _m_cnn.add(Input(shape=(SEQ_LEN, NUM_FEATS)))
    _m_cnn.add(Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'))
    _m_cnn.add(MaxPooling1D(pool_size=2))
    _m_cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'))
    _m_cnn.add(LSTM(64, return_sequences=False))
    _m_cnn.add(Dropout(0.3))
    _m_cnn.add(Dense(32, activation='relu'))
    _m_cnn.add(Dense(1))
    _m_cnn.compile(loss=_loss2, optimizer='adam', metrics=['mae'])

    _hist_cnn = _m_cnn.fit(
        train_gen2,
        validation_data = val_gen2,
        epochs          = EPOCHS,
        callbacks       = [EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1,
        )],
        verbose=1,
    )

    # Batched prediction — avoids GPU OOM
    _test_gen_cnn = SequenceDataGenerator(
        X_test, X_test[:, 0, 0], batch_size=BATCH_SIZE, shuffle=False
    )
    _pred_cnn = _m_cnn.predict(_test_gen_cnn, verbose=0).flatten()[:len(X_test)]

    _m_res = save_benchmark_results(
        'cnn_lstm', HORIZON, y_test, _pred_cnn,
        BENCHMARK_DIR, log_transform=TARGET_LOG_TRANSFORM,
        extra_meta={'epochs_trained': len(_hist_cnn.history['loss'])}
    )
    _save_dl_plots(_hist_cnn, y_test, _pred_cnn, 'cnn_lstm')

    # Save model and training history
    _m_cnn.save(os.path.join(_models_dir, f'cnn_lstm_k{HORIZON}.h5'))
    with open(os.path.join(_models_dir, f'cnn_lstm_k{HORIZON}_history.json'), 'w') as _f:
        json.dump(
            {k: [float(v) for v in vals]
             for k, vals in _hist_cnn.history.items()}, _f, indent=2
        )

    _horizon_results['cnn_lstm'] = _m_res
    # FIXED: was _m["sMAPE"] — _m is undefined here, correct variable is _m_res
    print(f'       MAE={_m_res["MAE"]:.6f}  RMSE={_m_res["RMSE"]:.6f}  '
          f'R2={_m_res["R2"]:.4f}  sMAPE={_m_res["sMAPE"]:.4f}')

    del _m_cnn, _hist_cnn, _pred_cnn, _test_gen_cnn
    del train_gen, val_gen, train_gen2, val_gen2
    K.clear_session();  gc.collect()

    # ── Per-horizon summary CSV ──────────────────────────────────────────────
    _table = build_comparison_table(BENCHMARK_DIR, HORIZON)
    _csv   = os.path.join(_horizon_dir, 'summary_deep_learning.csv')
    _table.to_csv(_csv, index=False)
    print(f'\n  Summary CSV → {_csv}')

    # ── Run config JSON ──────────────────────────────────────────────────────
    with open(os.path.join(_horizon_dir, 'dl_run_config.json'), 'w') as _f:
        json.dump({
            'horizon_k'      : HORIZON,
            'log_transform'  : TARGET_LOG_TRANSFORM,
            'loss_function'  : LOSS_FUNCTION,
            'huber_delta'    : HUBER_DELTA,
            'batch_size'     : BATCH_SIZE,
            'epochs'         : EPOCHS,
            'patience'       : PATIENCE,
            'random_seed'    : RANDOM_SEED,
            'completed_at'   : datetime.now().isoformat(),
            'metrics'        : _horizon_results,
        }, _f, indent=4)

    all_results[HORIZON] = _horizon_results

    del X_train, y_train, X_val, y_val, X_test, y_test
    gc.collect()

    print(f'\n  DONE k={HORIZON}s')
    print('=' * 60)

_total_h = (time.time() - loop_start) / 3600
print(f'\nAll DL benchmarks complete — {_total_h:.2f} h total')

---
## Cross-Horizon Summary

In [ ]:
if not all_results:
    print('No results yet — run the main loop first.')
else:
    _rows = []
    for _h, _models in sorted(all_results.items()):
        for _name, _m in _models.items():
            _rows.append({'Horizon_k_s': _h, 'Model': _name, **_m})
    _df = pd.DataFrame(_rows)
    print('Deep Learning baselines — all horizons')
    print('=' * 72)
    print(_df.to_string(index=False, float_format='{:.6f}'.format))
    print('=' * 72)
    _csv = os.path.join(ANALYSIS_DIR, 'deep_learning_all_horizons.csv')
    _df.to_csv(_csv, index=False)
    print(f'Saved → {_csv}')